# LangChain LCEL: Sequential State Management

This notebook demonstrates how to build robust **Multi-Step Pipelines** using LangChain Expression Language (LCEL). It focuses on using `RunnablePassthrough.assign` to maintain and augment state across an execution graph.

### Key Workflows
* **Atomic Recipe Generator:** A linear chain that transforms a location into a dish, recipe, and estimated cooking time.
* **Sentiment Analysis Pipeline:** A real-world example of processing product reviews into sentiment, summaries, and automated customer responses.

In [2]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

import os
from langchain_groq import ChatGroq

from dotenv import load_dotenv
load_dotenv()

# Initialize the Groq model cleanly
llama_llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=256
)

## Chains

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import json

In [4]:
# ---------------------------------------------------------
# Define the Atomic Prompts
# ---------------------------------------------------------
prompt_location = PromptTemplate.from_template(
    "Name exactly one classic dish from {location}. Do not include any other text."
)

prompt_dish = PromptTemplate.from_template(
    "Write a very short, 3-step recipe for {meal}."
)

prompt_time = PromptTemplate.from_template(
    "Based on this recipe, strictly estimate the total cooking time: {recipe}"
)

In [5]:
# ---------------------------------------------------------
# Build the Atomic Chains (The 'Workers')
# ---------------------------------------------------------
# Each chain is a simple, isolated execution graph.
chain_location = prompt_location | llama_llm | StrOutputParser()
chain_dish = prompt_dish | llama_llm | StrOutputParser()
chain_time = prompt_time | llama_llm | StrOutputParser()

In [6]:
# ---------------------------------------------------------
# The Master Execution Graph (The 'Manager')
# ---------------------------------------------------------
master_chain = (
    # Step 1: Execute chain_location, map output to 'meal'
    RunnablePassthrough.assign(meal=chain_location)
    
    # Step 2: Execute chain_dish, map output to 'recipe'
    | RunnablePassthrough.assign(recipe=chain_dish)
    
    # Step 3: Execute chain_time, map output to 'time'
    | RunnablePassthrough.assign(time=chain_time)
)

In [7]:
# ---------------------------------------------------------
# Execution
# ---------------------------------------------------------
print("Executing Master Chain...\n")
# We start the graph by feeding it the initial dictionary state
final_state = master_chain.invoke({"location": "China"})

# Print the beautifully accumulated JSON state
print(json.dumps(final_state, indent=4))

Executing Master Chain...

{
    "location": "China",
    "meal": "Kung Pao Chicken.",
    "recipe": "Here's a 3-step recipe for Kung Pao Chicken:\n\n**Servings:** 4\n**Prep Time:** 15 minutes\n**Cook Time:** 20 minutes\n\n**Ingredients:**\n\n- 1 lb boneless, skinless chicken breasts, cut into bite-sized pieces\n- 2 tablespoons vegetable oil\n- 1 cup roasted peanuts\n- 1 cup mixed vegetables (bell peppers, carrots, scallions)\n- 2 cloves garlic, minced\n- 1 tablespoon soy sauce\n- 1 tablespoon Shaoxing wine (or dry sherry)\n- 1 teaspoon cornstarch\n- Salt and pepper to taste\n- 1-2 dried red chili peppers, crushed or 1-2 teaspoons red pepper flakes\n\n**Recipe:**\n\n1. **Marinate and Cook Chicken:** In a bowl, whisk together soy sauce, Shaoxing wine, and cornstarch. Add the chicken and mix well. Heat 1 tablespoon of vegetable oil in a wok or large skillet over high heat. Add the chicken and cook until browned, about 5-6 minutes. Remove the chicken from the wok and set aside.\n\n2. **St

## Multi-Step Pipeline

In [8]:
# ---------------------------------------------------------
# Sample Data
# ---------------------------------------------------------
positive_review = """I absolutely love this coffee maker! It brews quickly and the coffee tastes amazing. 
The built-in grinder saves me so much time in the morning, and the programmable timer means 
I wake up to fresh coffee every day. Worth every penny and highly recommended to any coffee enthusiast."""

negative_review = """Disappointed with this laptop. It's constantly overheating after just 30 minutes of use, 
and the battery life is nowhere near the 8 hours advertised - I barely get 3 hours. 
The keyboard has already started sticking on several keys after just two weeks. Would not recommend to anyone."""

# ---------------------------------------------------------
# 3. Define the Prompt Templates
# ---------------------------------------------------------
sentiment_prompt = PromptTemplate.from_template(
    """Analyze the sentiment of the following product review as positive, negative, or neutral.
    Provide your analysis in exactly one word.
    Review: {review}
    Sentiment:"""
)

summary_prompt = PromptTemplate.from_template(
    """Summarize the following product review into 3 short bullet points.
    Review: {review}
    Sentiment: {sentiment}
    Summary:"""
)

response_prompt = PromptTemplate.from_template(
    """Write a short, professional response to a customer based on their review.
    If the sentiment is positive, thank them. If negative, apologize and offer a solution.
    Review: {review}
    Sentiment: {sentiment}
    Key points: {summary}
    Response:"""
)

# ---------------------------------------------------------
# Build the Atomic Chains (The Workers)
# ---------------------------------------------------------
chain_sentiment = sentiment_prompt | llama_llm | StrOutputParser()
chain_summary = summary_prompt | llama_llm | StrOutputParser()
chain_response = response_prompt | llama_llm | StrOutputParser()

# ---------------------------------------------------------
# The Master Execution Graph (Pure LCEL)
# ---------------------------------------------------------
master_chain = (
    # Step 1: Input state is {"review": text}
    # chain_sentiment automatically finds {review} in the dict.
    RunnablePassthrough.assign(sentiment=chain_sentiment)
    
    # Step 2: Input state is now {"review": text, "sentiment": text}
    # chain_summary automatically finds {review} and {sentiment}.
    | RunnablePassthrough.assign(summary=chain_summary)
    
    # Step 3: Input state is now {"review": text, "sentiment": text, "summary": text}
    # chain_response automatically finds all three variables.
    | RunnablePassthrough.assign(response=chain_response)
)

# ---------------------------------------------------------
# Execution & Testing
# ---------------------------------------------------------
def run_pipeline(review_text):
    print("\n" + "="*70)
    print("EXECUTING PIPELINE...")
    
    # We trigger the graph with the initial state dictionary
    final_state = master_chain.invoke({"review": review_text})
    
    # Print the accumulated state as a clean JSON
    print(json.dumps(final_state, indent=4))

In [9]:
run_pipeline(positive_review)


EXECUTING PIPELINE...
{
    "review": "I absolutely love this coffee maker! It brews quickly and the coffee tastes amazing. \nThe built-in grinder saves me so much time in the morning, and the programmable timer means \nI wake up to fresh coffee every day. Worth every penny and highly recommended to any coffee enthusiast.",
    "sentiment": "Positive.",
    "summary": "Here are 3 short bullet points summarizing the product review:\n\n * The coffee maker brews coffee quickly and produces great-tasting coffee.\n * The built-in grinder saves time in the morning and the programmable timer ensures fresh coffee every day.\n * The product is worth the investment and highly recommended for coffee enthusiasts.",
    "response": "Dear valued customer,\n\nWe are thrilled to hear that you're enjoying your coffee maker and experiencing the benefits of its quick brewing, great-tasting coffee, and convenient features like the built-in grinder and programmable timer. We're glad to know that it's maki

In [10]:
run_pipeline(negative_review)


EXECUTING PIPELINE...
{
    "review": "Disappointed with this laptop. It's constantly overheating after just 30 minutes of use, \nand the battery life is nowhere near the 8 hours advertised - I barely get 3 hours. \nThe keyboard has already started sticking on several keys after just two weeks. Would not recommend to anyone.",
    "sentiment": "Negative.",
    "summary": "Here are 3 short bullet points summarizing the product review:\n\n* The laptop constantly overheats after 30 minutes of use.\n* The battery life is significantly shorter than advertised, lasting only 3 hours.\n* The keyboard starts to stick on several keys after just two weeks of use.",
    "response": "Dear valued customer,\n\nThank you for taking the time to share your concerns about your laptop experience. We apologize for the issues you've encountered, including overheating, poor battery life, and keyboard malfunction. These problems are not the standards we strive to meet, and we're truly sorry for the inconveni